# 01 - Data Loading and Cleaning

**Project:** Land-Use Efficiency and Crop Yield Trends in the European Union  
**Author:** [Alireza Hosseini]
**Date:** 2026  
**Data source:** FAO. 2026. *FAOSTAT Statistical Database*. Accessed 8 January 2026. https://www.fao.org/faostat/. Licence: CC BY-4.0.

---

## Purpose

This notebook prepares the FAOSTAT raw data for the subsequent analysis. It conducts three essential tasks:

1. **Filtering** the global FAOSTAT export to six EU countries (Germany, France, Italy, Spain, Portugal, Netherlands) and four crops (Wheat, Maize, Grapes, Sugar beet).
2. **Cleaning** the data by dropping redundant administrative columns, normalizing country names, coercing numeric columns, and imputing missing values within each country-crop-element group.
3. **Reshaping** the table by pivoting Production and Area harvested into a wide format and computing Annual Yield (t/ha) as Production divided by Area.

The output (`eu_stat_joined.csv`) is utilized by notebooks 02 to 04.

## Research Questions

1. How have crop yields evolved in the EU over the last 30 years?
2. Which EU countries achieve higher yields without expanding agricultural land?
3. Are Southern EU countries converging or diverging from Northern EU yield levels?


## 1. Imports and configuration

All imports and path constants are declared at the top of the notebook in compliance with PEP 8. The paths are expressed relative to the `notebooks/` directory; therefore, the project remains portable across machines.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# Project paths (relative to notebooks/)
DATA_DIR = Path("../data")
RAW_FILE = DATA_DIR / "raw" / "FAOSTAT_data_en_1-8-2026.csv"
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Analytical scope
EU_COUNTRIES = [
    "France", "Germany", "Italy", "Spain", "Portugal",
    "Netherlands (Kingdom of the)",
]
CROPS = ["Wheat", "Maize (corn)", "Grapes", "Sugar beet"]


## 2. Load raw FAOSTAT data

The CSV file is read, and the success of the loading process is immediately verified by inspecting the shape, dtypes, and a sample of rows. This defensive check is essential as it identifies malformed files before any subsequent processing.


In [2]:
faostat = pd.read_csv(RAW_FILE)

print(f"Shape: {faostat.shape}")
print(f"Columns: {list(faostat.columns)}")
faostat.head()


Shape: (10800, 15)
Columns: ['Domain Code', 'Domain', 'Area Code (M49)', 'Area', 'Element Code', 'Element', 'Item Code (CPC)', 'Item', 'Year Code', 'Year', 'Unit', 'Value', 'Flag', 'Flag Description', 'Note']


,Domain Code,Domain,Area Code (M49),Area,Element Code,Element,Item Code (CPC),Item,Year Code,Year,Unit,Value,Flag,Flag Description,Note
0,QCL,Crops and livestock products,8,Albania,5312,Area harvested,1330,Grapes,1995,1995,ha,4342.0,A,Official figure,NaN
1,QCL,Crops and livestock products,8,Albania,5510,Production,1330,Grapes,1995,1995,t,55493.0,A,Official figure,NaN
2,QCL,Crops and livestock products,8,Albania,5312,Area harvested,1330,Grapes,1996,1996,ha,4345.0,A,Official figure,NaN
3,QCL,Crops and livestock products,8,Albania,5510,Production,1330,Grapes,1996,1996,t,59115.0,A,Official figure,NaN
4,QCL,Crops and livestock products,8,Albania,5312,Area harvested,1330,Grapes,1997,1997,ha,4121.0,A,Official figure,NaN


### Quick data inventory

A brief overview of the unique values in the categorical columns confirms the contents of the dataset prior to filtering.


In [3]:
print(f"Unique countries (Areas): {faostat['Area'].nunique()}")
print(f"Unique items (crops):        {faostat['Item'].nunique()}")
print(f"Unique elements:             {faostat['Element'].unique()}")
print(f"Year range:                  {faostat['Year'].min()} - {faostat['Year'].max()}")


Unique countries (Areas): 45
Unique items (crops):        4
Unique elements:             <StringArray>
['Area harvested', 'Production']
Length: 2, dtype: str
Year range:                  1995 - 2024


## 3. Filter to scope

The dataset is restricted to the six EU countries and four crops defined in the project scope. Administrative columns that carry no analytical value (codes, flags, notes) are dropped. The `errors="ignore"` argument prevents failure if any of these columns are absent in a future FAOSTAT export. This is a defensive choice for reproducibility.


In [4]:
COLS_TO_DROP = [
    "Domain", "Domain Code", "Note",
    "Area Code (M49)", "Element Code", "Item Code (CPC)",
    "Unnamed: 0", "Year Code", "Flag", "Flag Description", "Unit",
]

eu_stat = (
    faostat
    .loc[faostat["Area"].isin(EU_COUNTRIES) & faostat["Item"].isin(CROPS)]
    .drop(columns=COLS_TO_DROP, errors="ignore")
    .copy()
)

print(f"Filtered shape: {eu_stat.shape}")
eu_stat.head()


Filtered shape: (1440, 5)


,Area,Element,Item,Year,Value
3360,France,Area harvested,Grapes,1995,894800.0
3361,France,Production,Grapes,1995,7212900.0
3362,France,Area harvested,Grapes,1996,886690.0
3363,France,Production,Grapes,1996,7716400.0
3364,France,Area harvested,Grapes,1997,878148.0


## 4. Standardise country names

FAOSTAT uses *Netherlands (Kingdom of the)* as the official country label. To enhance readability in plots and tables, this label is shortened to *Netherlands*.


In [5]:
eu_stat["Area"] = eu_stat["Area"].replace(
    "Netherlands (Kingdom of the)", "Netherlands"
)


## 5. Coerce numeric column

The `Value` column is read as object dtype because some FAOSTAT cells contain non-numeric flags. Coercing with `errors="coerce"` converts unparseable entries to `NaN`, which are subsequently handled in the following step.


In [6]:
eu_stat["Value"] = pd.to_numeric(eu_stat["Value"], errors="coerce")

n_missing = eu_stat["Value"].isna().sum()
print(f"Numeric values missing after coercion: {n_missing}")


Numeric values missing after coercion: 6


## 6. Handle missing values

Missing values are imputed within each country-crop-element group rather than across the entire table. This approach preserves the integrity of each time series. Specifically, a missing French wheat yield is filled from the neighbouring French wheat values, and not from any other country or crop.

Three steps are conducted in order:

1. `interpolate()` performs linear interpolation between known values.
2. `ffill()` performs forward-fill for missing values at the *end* of a series.
3. `bfill()` performs backward-fill for missing values at the *start* of a series.

This combination ensures that every row receives a value while leaving genuine internal observations untouched.


In [7]:
eu_stat = eu_stat.sort_values(["Area", "Item", "Element", "Year"])

eu_stat["Value"] = (
    eu_stat
    .groupby(["Area", "Item", "Element"])["Value"]
    .transform(lambda s: s.interpolate().ffill().bfill())
)

eu_stat = eu_stat.drop_duplicates()

print(f"Remaining missing values: {eu_stat['Value'].isna().sum()}")
print(f"Final shape: {eu_stat.shape}")


Remaining missing values: 0
Final shape: (1440, 5)


## 7. Save cleaned long-format data

The cleaned long-format table is written as a checkpoint. Storing intermediate outputs is beneficial because it makes the subsequent notebooks runnable without re-executing the full pipeline.


In [8]:
cleaned_path = PROCESSED_DIR / "eu_stat_cleaned.csv"
eu_stat.to_csv(cleaned_path, index=False)
print(f"Saved: {cleaned_path}")


Saved: ..\data\processed\eu_stat_cleaned.csv


## 8. Reshape to wide format and compute yield

The cleaned table is in long format with one row per (country, crop, year, element). For analytical purposes, it is more convenient to have **Production** and **Area harvested** as separate columns so that yield can be computed as a single arithmetic operation.

The steps are as follows:

1. Splitting the long table into two by element.
2. Renaming the `Value` column in each (`production_t`, `area_ha`).
3. Merging the two on (Area, Item, Year), and validating that the relationship is one-to-one. Any other cardinality would indicate a data integrity issue.
4. Computing **Annual Yield (t/ha)** as Production divided by Area.

Additionally, a defensive `replace(0, pd.NA)` on `area_ha` prevents division-by-zero from producing infinities.


In [9]:
production = (
    eu_stat[eu_stat["Element"] == "Production"]
    .rename(columns={"Value": "production_t"})
    [["Area", "Item", "Year", "production_t"]]
)

harvest_area = (
    eu_stat[eu_stat["Element"] == "Area harvested"]
    .rename(columns={"Value": "area_ha"})
    [["Area", "Item", "Year", "area_ha"]]
)

eu_stat_joined = production.merge(
    harvest_area,
    on=["Area", "Item", "Year"],
    how="left",
    validate="one_to_one",
)

eu_stat_joined["area_ha"] = eu_stat_joined["area_ha"].replace(0, pd.NA)
eu_stat_joined["Annual Yield (t/ha)"] = (
    eu_stat_joined["production_t"] / eu_stat_joined["area_ha"]
)
eu_stat_joined = eu_stat_joined.sort_values(
    ["Area", "Item", "Year"]
).reset_index(drop=True)

eu_stat_joined.head()


,Area,Item,Year,production_t,area_ha,Annual Yield (t/ha)
0,France,Grapes,1995,7212900.0,894800.0,8.060907
1,France,Grapes,1996,7716400.0,886690.0,8.702478
2,France,Grapes,1997,7190900.0,878148.0,8.188711
3,France,Grapes,1998,6919069.0,862677.0,8.020463
4,France,Grapes,1999,8040710.0,863355.0,9.31333


## 9. Save joined dataset

This file serves as the input for notebooks 02, 03, and 04.


In [10]:
joined_path = PROCESSED_DIR / "eu_stat_joined.csv"
eu_stat_joined.to_csv(joined_path, index=False)
print(f"Saved: {joined_path}")
print(f"Shape: {eu_stat_joined.shape}")


Saved: ..\data\processed\eu_stat_joined.csv
Shape: (720, 6)


## Summary

The pipeline produced two artifacts:

- `data/processed/eu_stat_cleaned.csv`: cleaned long-format data with imputed missing values.
- `data/processed/eu_stat_joined.csv`: wide-format data with computed annual yield.

Six countries, four crops, thirty years, and multiple elements yield approximately 720 production-area observations covering 1995 to 2024. All numeric values were coerced and the gaps within each country-crop-element series were interpolated. Consequently, the subsequent notebooks can rely on a complete time series.
